# 01 — Data Collection

Fetch all raw data via `nba_api` and cache to `data/raw/` as parquet files.

**Output:** `data/raw/playoff_games_{season}.parquet`, `data/raw/team_metrics_{season}.parquet`

**Note on rest days:** Rest days are computed from the playoff game dates themselves (no separate game log fetch needed).

**Run time:** ~5–10 min first run; subsequent runs read from cache instantly.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from tqdm import tqdm

sys.path.insert(0, str(Path().resolve().parent))
from src.data import fetch_playoff_games, fetch_team_estimated_metrics

In [2]:
# CONFIG — modify here only
SEASONS = [
    "2014-15", "2015-16", "2016-17", "2017-18", "2018-19",
    "2020-21", "2021-22", "2022-23", "2023-24",
]
FORCE_REFETCH = False  # set True to ignore cache and re-download

## 1. Fetch Playoff Game Logs

In [3]:
all_games = []
for season in tqdm(SEASONS, desc="Playoff games"):
    df = fetch_playoff_games(season, force=FORCE_REFETCH)
    df["season"] = season
    all_games.append(df)

games = pd.concat(all_games, ignore_index=True)
print(f"{len(games)} total playoff game rows across {games['season'].nunique()} seasons")
games.head()

Playoff games:   0%|          | 0/9 [00:00<?, ?it/s]

Playoff games: 100%|██████████| 9/9 [00:00<00:00, 96.24it/s]

1496 total playoff game rows across 9 seasons


,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,...,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS,season
0,42014,1610612744,GSW,Golden State Warriors,0041400406,2015-06-16,GSW @ CLE,W,239,105,...,7,32,39,28,11,4,9,27,8.0,2014-15
1,42014,1610612739,CLE,Cleveland Cavaliers,0041400406,2015-06-16,CLE vs. GSW,L,239,97,...,16,40,56,14,3,7,16,26,-8.0,2014-15
2,42014,1610612744,GSW,Golden State Warriors,0041400405,2015-06-14,GSW vs. CLE,W,239,104,...,11,32,43,25,7,2,16,25,13.0,2014-15
3,42014,1610612739,CLE,Cleveland Cavaliers,0041400405,2015-06-14,CLE @ GSW,L,241,91,...,10,27,37,17,10,4,10,28,-13.0,2014-15
4,42014,1610612739,CLE,Cleveland Cavaliers,0041400404,2015-06-11,CLE vs. GSW,L,239,82,...,16,33,49,16,2,3,9,19,-21.0,2014-15


## 2. Fetch Team Estimated Metrics (ORtg, DRtg, pace)

In [4]:
all_metrics = []
for season in tqdm(SEASONS, desc="Team metrics"):
    df = fetch_team_estimated_metrics(season, force=FORCE_REFETCH)
    df["season"] = season
    all_metrics.append(df)

metrics = pd.concat(all_metrics, ignore_index=True)
print(f"{len(metrics)} team-season rows")
metrics.head()

Team metrics:   0%|          | 0/9 [00:00<?, ?it/s]

Team metrics: 100%|██████████| 9/9 [00:00<00:00, 773.78it/s]

270 team-season rows


,TEAM_NAME,TEAM_ID,GP,W,L,W_PCT,MIN,E_OFF_RATING,E_DEF_RATING,E_NET_RATING,...,E_OFF_RATING_RANK,E_DEF_RATING_RANK,E_NET_RATING_RANK,E_AST_RATIO_RANK,E_OREB_PCT_RANK,E_DREB_PCT_RANK,E_REB_PCT_RANK,E_TM_TOV_PCT_RANK,E_PACE_RANK,season
0,Golden State Warriors,1610612744,82,67,15,0.817,3946.0,109.7,99.6,10.1,...,2,1,1,1,21,18,12,14,1,2014-15
1,Houston Rockets,1610612745,82,56,26,0.683,3961.0,104.2,100.8,3.4,...,12,6,6,16,7,28,15,28,2,2014-15
2,Denver Nuggets,1610612743,82,30,52,0.366,3976.0,101.6,105.2,-3.6,...,21,26,24,21,9,17,18,11,3,2014-15
3,Phoenix Suns,1610612756,82,39,43,0.476,3976.0,102.9,103.8,-0.9,...,14,17,19,29,19,21,25,20,3,2014-15
4,Boston Celtics,1610612738,82,40,42,0.488,3976.0,101.7,101.6,0.2,...,20,14,18,6,16,14,20,6,5,2014-15


## 3. Validation

In [5]:
print("Playoff game rows per season (2 rows per game — one per team):")
print(games.groupby("season").size().to_string())
print()
print(f"Columns: {list(games.columns)}")
print()
print("Team metrics columns:")
print(metrics.columns.tolist())
print()
print("Missing values in games:")
print(games.isnull().sum()[games.isnull().sum() > 0])

Playoff game rows per season (2 rows per game — one per team):
season
2014-15    162
2015-16    172
2016-17    158
2017-18    164
2018-19    164
2020-21    170
2021-22    174
2022-23    168
2023-24    164

Columns: ['SEASON_ID', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME', 'GAME_ID', 'GAME_DATE', 'MATCHUP', 'WL', 'MIN', 'PTS', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PLUS_MINUS', 'season']

Team metrics columns:
['TEAM_NAME', 'TEAM_ID', 'GP', 'W', 'L', 'W_PCT', 'MIN', 'E_OFF_RATING', 'E_DEF_RATING', 'E_NET_RATING', 'E_PACE', 'E_AST_RATIO', 'E_OREB_PCT', 'E_DREB_PCT', 'E_REB_PCT', 'E_TM_TOV_PCT', 'GP_RANK', 'W_RANK', 'L_RANK', 'W_PCT_RANK', 'MIN_RANK', 'E_OFF_RATING_RANK', 'E_DEF_RATING_RANK', 'E_NET_RATING_RANK', 'E_AST_RATIO_RANK', 'E_OREB_PCT_RANK', 'E_DREB_PCT_RANK', 'E_REB_PCT_RANK', 'E_TM_TOV_PCT_RANK', 'E_PACE_RANK', 'season']

Missing values in games:
Series([], dtype: int64)
